# Nagao Adaptive Edge-Preserving Filter (TP 2 — Exercise 3)

## Assignment Objectives
Implement and evaluate `filtreNagao(im, taille)` for $5\times5$, $7\times7$
and $9\times9$ windows on `im2` (Gaussian noise) and `im3` (impulsive noise),
comparing PSNR against the classical filters of the previous exercise.

## Mathematical Principles & Intuition
The Nagao filter slices the $k \times k$ neighbourhood of each pixel into
**9 overlapping sub-regions** (cliques): one central square, 4 cardinal and
4 diagonal wedges. For each clique $S_i$ it computes

$$\mu_i = \frac{1}{|S_i|}\sum_{p \in S_i} I(p), \qquad
\sigma_i^2 = \frac{1}{|S_i|}\sum_{p \in S_i} \big(I(p) - \mu_i\big)^2$$

and replaces the pixel by $\mu_{i^\star}$ with $i^\star = \arg\min_i \sigma_i^2$ —
the mean of the **most homogeneous** clique.

### Why it preserves edges
Any clique straddling a contour mixes two intensity populations, so its
variance explodes; the minimum-variance clique lies entirely on one side of
the edge. The output is therefore smoothed *along* edges, never *across*
them — the visual result approaches a cartoon/painting effect.

### Implementation notes
`filtreNagao` (in `TAI/utils/filters.py`) is fully vectorised: clique means
and variances are computed with `cv2.filter2D` on the image and its square,
then `argmin` picks the winning clique per pixel.


---
## Step 0: Portable Environment Bootstrap (Local IDE / Jupyter + Google Colab)

### What This Cell Does
- Detects Google Colab (`google.colab` in `sys.modules` or `COLAB_RELEASE_TAG`); on Colab it clones `APSD_TAI_LAB` into `/content` (unless already mounted from Drive), so the notebook works even when opened directly from GitHub.
- Locally, it walks upward from the current working directory to the repository root (marker: `README.md` + `APDS/` + `TAI/`) and registers it on `sys.path` — the notebook therefore runs identically from the repo root, from `TAI/…/notebooks/`, or from any sub-folder.
- Verifies `opencv-contrib-python` (checking `cv2.ximgproc`) and installs it only when missing.
- Calls `bootstrap()` from `TAI.utils.env_io`, which resolves `TAI/images/` (the single consolidated, SHA-256 deduplicated dataset) and `TAI/outputs/` (runtime artifacts, never committed).

### Why It Is Needed
- `cv2.imshow()` opens blocking GUI windows and freezes Jupyter kernels — everything here renders through inline Matplotlib instead.
- No hardcoded absolute paths (like `/home/...`): every path is derived from the discovered repository root, so the same cell works on any machine, in any IDE, and on Colab.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Vtheonly/APSD_TAI_LAB.git"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    # Google Colab: clone the repository (unless already present, e.g. on Drive)
    _repo = Path("/content/APSD_TAI_LAB")
    if not (_repo / "TAI" / "utils").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True
        )
else:
    # Local IDE / Jupyter: walk upward to the repository root
    # (identified by README.md + APDS/ + TAI/), whatever the launch directory.
    _cwd = Path.cwd().resolve()
    _repo = next(
        (
            c
            for c in (_cwd, *_cwd.parents)
            if (c / "README.md").is_file() and (c / "APDS").is_dir() and (c / "TAI").is_dir()
        ),
        None,
    )
    if _repo is None:
        raise FileNotFoundError(
            f"APSD_TAI_LAB repository root not found above {_cwd}. "
            "Launch Jupyter from inside the cloned repository."
        )
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))

# Make sure opencv-contrib-python is used (cv2.ximgproc) -- install only if missing
try:
    import cv2

    _has_ximgproc = hasattr(cv2, "ximgproc")
except ImportError:
    _has_ximgproc = False
if not _has_ximgproc:
    %pip install -q opencv-contrib-python
    import cv2

from TAI.utils.env_io import bootstrap, load_image, show_images, plot_image_with_hist

ROOT_DIR, IMG_DIR, OUT_DIR = bootstrap()

import math
import time

import matplotlib.pyplot as plt
import numpy as np

print(f"Repository root : {ROOT_DIR}")
print(f"Images folder   : {IMG_DIR} ({len(list(IMG_DIR.iterdir()))} images)")
print(f"Outputs folder  : {OUT_DIR}")
print(f"OpenCV          : {cv2.__version__} | cv2.ximgproc available: {hasattr(cv2, 'ximgproc')}")


---
## Step 1: Deterministic Inputs and the 9 Cliques


In [ ]:
from TAI.utils.filters import build_nagao_masks, filtreNagao
from TAI.utils.metrics import calculate_psnr
from TAI.utils.noise import add_gaussian_noise, sp_noise

im1 = load_image("tp1_tp2_cameraman_gray.bmp", cv2.IMREAD_GRAYSCALE)

gauss_path = OUT_DIR / "tp2_exo1_cameraman_gauss_var100.bmp"
sp_path = OUT_DIR / "tp2_exo1_cameraman_salt_pepper_p10.bmp"
im2 = cv2.imread(str(gauss_path), cv2.IMREAD_GRAYSCALE) if gauss_path.exists() else add_gaussian_noise(im1, 100.0, seed=42)
im3 = cv2.imread(str(sp_path), cv2.IMREAD_GRAYSCALE) if sp_path.exists() else sp_noise(im1, 0.10, seed=42)

# Inspect the 9 sub-masks of the 5x5 Nagao window
masks5 = build_nagao_masks(5)
print(f"Number of cliques: {len(masks5)} | clique sizes: {[int(m.sum()) for m in masks5]}")

fig, axes = plt.subplots(3, 3, figsize=(9, 9))
names = ["Centre", "N", "E", "S", "W", "NW", "SE", "SW", "NE"]
for ax, m, name in zip(axes.ravel(), masks5, names):
    ax.imshow(m, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{name} ({int(m.sum())} px)", fontsize=10)
    ax.axis("off")
plt.suptitle("The 9 Nagao sub-regions (5x5 window)", fontsize=13)
plt.tight_layout()
plt.show()


---
## Step 2: PSNR Benchmark for 5x5 / 7x7 / 9x9


In [ ]:
print(f"Baselines -> im2: {calculate_psnr(im1, im2):.3f} dB | im3: {calculate_psnr(im1, im3):.3f} dB\n")
print(f"{'Nagao window':16s} | {'PSNR im2 (Gaussian)':>20s} | {'PSNR im3 (S&P)':>14s}")
print("-" * 58)

results_im2, results_im3 = [], []
for sz in (5, 7, 9):
    t0 = time.perf_counter()
    out2 = filtreNagao(im2, sz)
    dt2 = (time.perf_counter() - t0) * 1000.0
    out3 = filtreNagao(im3, sz)
    results_im2.append((f"Nagao {sz}x{sz} on im2 ({calculate_psnr(im1, out2):.2f} dB, {dt2:.0f} ms)", out2))
    results_im3.append((f"Nagao {sz}x{sz} on im3 ({calculate_psnr(im1, out3):.2f} dB)", out3))
    print(f"{f'{sz}x{sz}':16s} | {calculate_psnr(im1, out2):18.3f} dB | {calculate_psnr(im1, out3):12.3f} dB")

out2_iter3 = filtreNagao(im2, 5, iterations=3)
print(f"{'5x5 (3 iter.)':16s} | {calculate_psnr(im1, out2_iter3):18.3f} dB | {'-':>14s}")

show_images([("Original im1", im1)] + results_im2, cols=4)
show_images([("S&P noise im3", im3)] + results_im3, cols=4)


In [ ]:
# Edge preservation zoom: compare mean 5x5 vs Nagao 5x5 on a detail crop
mean5 = cv2.blur(im2, (5, 5))
nagao5 = filtreNagao(im2, 5)
crop = lambda im: im[70:200, 70:200]

show_images([
    ("Original (zoom)", crop(im1)),
    ("Gaussian noise (zoom)", crop(im2)),
    ("Mean 5x5 (zoom)", crop(mean5)),
    ("Nagao 5x5 (zoom)", crop(nagao5)),
], cols=4)


## Analysis & Key Takeaways
- **On Gaussian noise (`im2`):** Nagao $5\times5$ clearly beats the plain mean filter at equal window size while keeping contours razor-sharp — the "cartoon" rendering is the visual signature of edge-aligned smoothing. Larger windows ($7\times7$, $9\times9$) enlarge the cliques, washing out fine detail and slightly reducing PSNR; iterating 3x a small Nagao is usually better than one large pass.
- **On salt & pepper (`im3`):** Nagao automatically avoids cliques containing an aberrant pixel (their variance explodes) as long as at least one healthy clique surrounds the pixel. But when the *centre pixel itself* is corrupted, every clique contains it — the median filter remains the reference for pure impulse noise.
- **Complexity:** the vectorised implementation costs 9 convolutions per iteration (one per clique) instead of 9 nested loops — ~2 orders of magnitude faster than a naive Python traversal.
